# 04 — Benchmark: XGBoost

**ML Ensemble Benchmarking Framework**

This notebook applies feature selection and class imbalance handling to the engineered features, then runs automated hyperparameter tuning via `GridSearchCV` for **XGBoost**, and evaluates the tuned model on the held-out test set.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data.load_data import load_raw_data, train_test_split_data
from src.data.preprocess import preprocess_pipeline
from src.features.build_features import engineer_features
from src.features.selection import select_features
from src.features.imbalance import apply_smote
from src.models.xgboost_model import XGBoostModel
from src.tuning.grid_search import tune_model
from src.evaluation.metrics import compute_classification_metrics

RANDOM_STATE = 42

## Load, Preprocess, and Engineer Features

Repeats the pipeline from `02_feature_engineering.ipynb` so this notebook is runnable standalone.

In [ ]:
df = load_raw_data(n_samples=8000, n_features=20)
X_train, X_test, y_train, y_test = train_test_split_data(df, test_size=0.2)

X_train_pp, X_test_pp, _ = preprocess_pipeline(X_train, X_test)
X_train_fe = engineer_features(X_train_pp, max_interaction_pairs=10)
X_test_fe = engineer_features(X_test_pp, max_interaction_pairs=10)

print(f"Engineered shape: train={X_train_fe.shape}, test={X_test_fe.shape}")

## Feature Selection

Selects the top-20 features by Random Forest importance, fit on training data only.

In [ ]:
X_train_sel, selected_cols = select_features(X_train_fe, y_train, method="importance", k=20)
X_test_sel = X_test_fe[selected_cols]

print(f"Selected {len(selected_cols)} features:")
print(selected_cols)

## Class Imbalance Handling (SMOTE)

Applied to the training split only — the test split is left untouched for an honest evaluation.

In [ ]:
X_train_bal, y_train_bal = apply_smote(X_train_sel, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_bal.value_counts().to_dict())

## Hyperparameter Tuning — GridSearchCV for XGBoost

In [ ]:
model = XGBoostModel()
print("Parameter grid:")
print(model.param_grid())

In [ ]:
result = tune_model(
    model,
    X_train_bal,
    y_train_bal,
    cv_folds=3,
    scoring="f1",
)

print(f"Best CV F1-score: {result.best_score:.4f}")
print(f"Best params: {result.best_params}")
print(f"Tuning time: {result.search_time_seconds:.2f}s")

## Evaluate on the Held-Out Test Set

In [ ]:
model.set_estimator(result.best_estimator)

y_pred = model.predict(X_test_sel)
y_proba = model.predict_proba(X_test_sel) if hasattr(model.estimator, "predict_proba") else None

metrics = compute_classification_metrics(y_test.to_numpy(), y_pred, y_proba)

print(f"Accuracy:  {metrics.accuracy:.4f}")
print(f"Precision: {metrics.precision:.4f}")
print(f"Recall:    {metrics.recall:.4f}")
print(f"F1-Score:  {metrics.f1:.4f}")
print(f"ROC-AUC:   {metrics.roc_auc:.4f}")
print(f"Confusion Matrix: {metrics.confusion_matrix}")

## Save the Tuned Model

In [ ]:
import os
os.makedirs("../models", exist_ok=True)
saved_path = model.save("../models")
print(f"Saved to {saved_path}")

## Summary

- **XGBoost** was tuned via `GridSearchCV` with Stratified 3-Fold cross-validation, optimizing for F1-score.
- The tuned model was evaluated once, on the held-out test set, after feature selection and SMOTE-based class balancing.
- Results are consistent with (though may vary slightly from, due to sampling) the reference numbers reported in `docs/results.md` and `README.md`.

**Next:** see the other per-model notebooks, then [`06_final_comparison.ipynb`](06_final_comparison.ipynb) for the head-to-head comparison and final visualizations.